In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance

In [ ]:
df = pd.read_csv("HRDataset_with_review_and_ancienete.csv")
df_raw = df.copy()

In [ ]:

cols_to_drop = [
    "Employee_Name",
    "EmpID",
    "GenderID",
    "Sex",
    "MaritalDesc",
    "CitizenDesc",
    "HispanicLatino",
    "RaceDesc",
    "MaritalStatusID",
    "MarriedID",
    "PositionID",
    "Zip",
    "DateofTermination",
    "TermReason",
    "EmploymentStatus",
    "Internal_Transfer_Request",
    "EmpStatusID",
    "Feedback_RH",
    "DateofHire",
    "LastPerformanceReview_Date"
]

cols_to_drop_existing = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop_existing)

In [ ]:
target_col = "Termd"

if target_col not in df.columns:
    raise ValueError(f"La colonne cible '{target_col}' est introuvable.")

# X = variables explicatives, y = cible
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)


In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Variables catégorielles :", categorical_features)
print("Variables numériques :", numeric_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",   # utile si départs vs non-départs déséquilibrés
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n===== ÉVALUATION =====")
print("ROC AUC :", roc_auc_score(y_test, y_proba))
print("\nClassification report :")
print(classification_report(y_test, y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))

In [ ]:
perm = permutation_importance(
    pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("\n===== TOP VARIABLES EXPLICATIVES DES DÉPARTS =====")
print(feature_importance.head(15))

In [ ]:
active_mask = df_raw["Termd"] == 0
active_employees_raw = df_raw.loc[active_mask].copy()

# On prépare les mêmes variables explicatives que pour l'entraînement
active_employees_X = active_employees_raw.drop(columns=cols_to_drop_existing, errors="ignore")
active_employees_X = active_employees_X.drop(columns=[target_col], errors="ignore")

# Prédiction du risque de départ
active_employees_raw["risk_of_leaving"] = pipeline.predict_proba(active_employees_X)[:, 1]

# Classement des employés les plus à risque
risk_columns_to_show = [
    col for col in [
        "Employee_Name", "Department", "Position", "ManagerName",
        "Salary", "EngagementSurvey", "EmpSatisfaction",
        "Absences", "DaysLateLast30", "ancienete", "TimeSinceLastReview_days"
    ] if col in active_employees_raw.columns
]

at_risk_employees = active_employees_raw[risk_columns_to_show + ["risk_of_leaving"]] \
    .sort_values("risk_of_leaving", ascending=False)

print("\n===== EMPLOYÉS ENCORE PRÉSENTS LES PLUS À RISQUE =====")
print(at_risk_employees.head(20))

# Sauvegarde optionnelle
feature_importance.to_csv("importance_variables_departs.csv", index=False)
at_risk_employees.to_csv("employes_a_risque_depart.csv", index=False)

print("\nFichiers sauvegardés :")
print("- importance_variables_departs.csv")
print("- employes_a_risque_depart.csv")